In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%bash
pip install kaggle
mkdir -p ~/.kaggle
cp /content/drive/MyDrive/Project/kaggle.json ~/.kaggle/
chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download nikanvasei/shanghaitech-campus-dataset

Dataset URL: https://www.kaggle.com/datasets/nikanvasei/shanghaitech-campus-dataset
License(s): MIT
100% 10.9G/10.9G [01:52<00:00, 104MB/s]



In [ ]:
%%bash
# install 7z library
wget https://www.7-zip.org/a/7z2301-linux-x64.tar.xz
tar -xf /content/7z2301-linux-x64.tar.xz

--2026-04-06 18:44:24--  https://www.7-zip.org/a/7z2301-linux-x64.tar.xz
Resolving www.7-zip.org (www.7-zip.org)... 89.167.91.206
Connecting to www.7-zip.org (www.7-zip.org)|89.167.91.206|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1527700 (1.5M) [application/octet-stream]
Saving to: ‘7z2301-linux-x64.tar.xz’

     0K .......... .......... .......... .......... ..........  3%  316K 5s
    50K .......... .......... .......... .......... ..........  6%  316K 4s
   100K .......... .......... .......... .......... .......... 10%  100M 3s
   150K .......... .......... .......... .......... .......... 13% 1.21M 2s
   200K .......... .......... .......... .......... .......... 16%  426K 2s
   250K .......... .......... .......... .......... .......... 20% 75.0M 2s
   300K .......... .......... .......... .......... .......... 23% 85.9M 2s
   350K .......... .......... .......... .......... .......... 26% 1.24M 1s
   400K .......... .......... .......... .........

In [ ]:
!/content/7zz x /content/shanghaitech-campus-dataset.zip


7-Zip (z) 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576, ASM

Scanning the drive for archives:
  0M Scan /content/                   1 file, 11718494278 bytes (11 GiB)

Extracting archive: /content/shanghaitech-campus-dataset.zip
  2% 4096 Open              --
Path = /content/shanghaitech-campus-dataset.zip
Type = zip
Physical Size = 11718494278
64-bit = +
Characteristics = Zip64

  0%      0% 196          0% 391          0% 542          0% 1016           0% 1292           0% 1517           1% 1784 - SHANGHAI/SHANGHAI_TRAIN/frames/01_0029/194.jpg                                                            1% 2349        

ٍ

In [ ]:
!ls /content/drive/MyDrive

Project


In [ ]:
!pip install ultralytics
import cv2
import os
from ultralytics import YOLO
from ultralytics.utils.plotting import colors, Annotator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
import os
import cv2
from ultralytics import YOLO
from ultralytics.utils.plotting import Annotator, colors


class ObjectTracking:
    def __init__(
        self,
        model="yolov8n.pt",
        source="frames/",
        output_txt="trajectory_output.txt",
        output_video="object-tracking.mp4",
    ):
        self.model = YOLO(model)
        self.frames_path = source
        self.output_txt = output_txt
        self.output_video = output_video

        self.image_files = sorted(
            [f for f in os.listdir(self.frames_path) if f.endswith((".jpg", ".png"))]
        )

        if not self.image_files:
            raise ValueError(f"No images found in: {self.frames_path}")

        first_img = cv2.imread(os.path.join(self.frames_path, self.image_files[0]))
        if first_img is None:
            raise ValueError(f"Could not read first image in: {self.frames_path}")

        h, w, _ = first_img.shape
        self.writer = cv2.VideoWriter(
            self.output_video,
            cv2.VideoWriter_fourcc(*"mp4v"),
            25,
            (w, h),
        )

    def run(self):
        with open(self.output_txt, "w") as f_out:
            for frame_idx, image_file in enumerate(self.image_files):
                img_path = os.path.join(self.frames_path, image_file)
                im0 = cv2.imread(img_path)

                if im0 is None:
                    print(f"Error reading image: {img_path}")
                    continue

                ann = Annotator(im0, line_width=3)

                results = self.model.track(
                    im0,
                    persist=True,
                    classes=[0],
                    verbose=False,
                    tracker="bytetrack.yaml",
                )

                if results and len(results) > 0:
                    result = results[0]

                    if result.boxes is not None and result.boxes.id is not None:
                        xyxy_boxes = result.boxes.xyxy.cpu()
                        xywh_boxes = result.boxes.xywh.cpu().tolist()
                        ids = result.boxes.id.cpu().tolist()
                        clss = result.boxes.cls.cpu().tolist()

                        for box_xyxy, box_xywh, obj_id, cls in zip(
                            xyxy_boxes, xywh_boxes, ids, clss
                        ):
                            xc, yc, w_box, h_box = box_xywh

                            f_out.write(
                                f"{frame_idx},{int(obj_id)},{xc:.2f},{yc:.2f},{w_box:.2f},{h_box:.2f}\n"
                            )

                            ann.box_label(
                                box_xyxy,
                                label=f"ID {int(obj_id)}",
                                color=colors(int(cls), True),
                            )

                annotated_frame = ann.result()
                self.writer.write(annotated_frame)

        self.writer.release()
        print(f"Processing Complete. Video saved as '{self.output_video}'")
        print(f"Trajectory data saved successfully in '{self.output_txt}'")


frames_root = "/content/SHANGHAI/SHANGHAI_TRAIN/frames"
output_root = "/content/tracking_outputs"

os.makedirs(output_root, exist_ok=True)

all_folders = sorted(
    [
        folder_name
        for folder_name in os.listdir(frames_root)
        if os.path.isdir(os.path.join(frames_root, folder_name))
    ]
)

print(f"Found {len(all_folders)} folders.")

for folder_name in all_folders:
    folder_path = os.path.join(frames_root, folder_name)

    output_txt = os.path.join(output_root, f"{folder_name}_trajectories.txt")
    output_video = os.path.join(output_root, f"{folder_name}_tracking.mp4")

    print(f"\nProcessing folder: {folder_name}")

    tracker = ObjectTracking(
        model="yolov8n.pt",
        source=folder_path,
        output_txt=output_txt,
        output_video=output_video,
    )
    tracker.run()

print("\nAll folders processed successfully.")


Found 238 folders.

Processing folder: 01_0014
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 233ms
Prepared 1 package in 61ms
Installed 1 package in 5ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Processing Complete. Video saved as '/content/tracking_outputs/01_0014_tracking.mp4'
Trajectory data saved successfully in '/content/tracking_outputs/01_0014_trajectories.txt'

Processing folder: 01_0016
Processing Complete. Video saved as '/content/tracking_outputs/01_0016_tracking.mp4'
Trajectory data saved successfully in '/content/tracking_outputs/01_0016_trajectories.txt'

Processing folder: 01_002
Processing Complete. Video saved as '/content/tracking_outputs/01_002_tracking.mp4'
Trajectory data saved successfully in '/content/tracking_outputs/01_002_trajectories.txt'

Processing fol

In [ ]:
import os
import numpy as np

txt_root = "/content/tracking_outputs"
labels_root = "/content/SHANGHAI/SHANGHAI_TRAIN/label"
output_root = "/content/tracking_with_labels"

os.makedirs(output_root, exist_ok=True)

for txt_file in os.listdir(txt_root):
    if not txt_file.endswith(".txt"):
        continue

    base_name = txt_file.replace("_trajectories.txt", "")
    txt_path = os.path.join(txt_root, txt_file)
    npy_path = os.path.join(labels_root, f"{base_name}.npy")
    output_path = os.path.join(output_root, txt_file)

    if not os.path.exists(npy_path):
        print(f"Missing label file for {base_name}")
        continue

    frame_labels = np.load(npy_path, allow_pickle=True)

    with open(txt_path, "r") as f:
        lines = f.readlines()

    with open(output_path, "w") as f:
        for line in lines:
            line = line.strip()
            if not line:
                continue

            parts = line.split(",")
            frame_id = int(parts[0])

            if frame_id >= len(frame_labels):
                print(f"Warning: frame {frame_id} exceeds label length for {base_name}")
                label = -1
            else:
                label = int(frame_labels[frame_id])

            f.write(f"{line},{label}\n")

    print(f"Saved: {output_path}")

print("All files processed.")


Saved: /content/tracking_with_labels/01_083_trajectories.txt
Saved: /content/tracking_with_labels/08_036_trajectories.txt
Saved: /content/tracking_with_labels/01_0140_trajectories.txt
Saved: /content/tracking_with_labels/08_039_trajectories.txt
Saved: /content/tracking_with_labels/08_0159_trajectories.txt
Saved: /content/tracking_with_labels/12_0143_trajectories.txt
Saved: /content/tracking_with_labels/01_025_trajectories.txt
Saved: /content/tracking_with_labels/01_010_trajectories.txt
Saved: /content/tracking_with_labels/06_004_trajectories.txt
Saved: /content/tracking_with_labels/05_066_trajectories.txt
Saved: /content/tracking_with_labels/04_0010_trajectories.txt
Saved: /content/tracking_with_labels/08_027_trajectories.txt
Saved: /content/tracking_with_labels/05_011_trajectories.txt
Saved: /content/tracking_with_labels/05_035_trajectories.txt
Saved: /content/tracking_with_labels/02_002_trajectories.txt
Saved: /content/tracking_with_labels/06_007_trajectories.txt
Saved: /content/trac

In [ ]:
import os
import zipfile

output_root = "/content/tracking_with_labels"
zip_path = "/content/tracking_txt_files.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_name in os.listdir(output_root):
        if file_name.endswith(".txt"):
            file_path = os.path.join(output_root, file_name)
            zipf.write(file_path, arcname=file_name)

print(f"Created zip file: {zip_path}")


Created zip file: /content/tracking_txt_files.zip


In [ ]:
from google.colab import files
files.download("/content/tracking_txt_files.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import numpy as np
x = np.load("/content/SHANGHAI/SHANGHAI_TRAIN/label/01_002.npy", allow_pickle=True)
print(x)
print(x.shape)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0]
(481,)
